In [12]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

In [13]:
df = pd.read_csv("../data/supply_chain_disruption_recovery.csv")

df = df.dropna()

# 1. Stronger response-time effect
df["production_impact_pct"] += (df["response_time_days"] / 64) * 15
df["full_recovery_days"] += (df["response_time_days"] / 64) * 40

# 2. Backup supplier penalty
mask = df["has_backup_supplier"] == "No"
df.loc[mask, "production_impact_pct"] += 12
df.loc[mask, "full_recovery_days"] += 25

# 3. Tier effect
tier_bonus = {1: 18, 2: 10, 3: 4, 4: 0}
df["production_impact_pct"] += df["supplier_tier"].map(tier_bonus)

# 4. Critical combination risk
critical = (df["supplier_tier"] == 1) & (df["has_backup_supplier"] == "No")
df.loc[critical, "production_impact_pct"] += 15
df.loc[critical, "full_recovery_days"] += 35

# 5. Fixed Revenue Loss Calculation
# (Note: Defining a basic multiplier mapping based on tier so it runs perfectly)
tier_multipliers = {1: 1.5, 2: 1.2, 3: 1.0, 4: 0.8}
supplier_multiplier = df["supplier_tier"].map(tier_multipliers)

df["revenue_loss_usd"] = (
    df["production_impact_pct"] * df["full_recovery_days"] * supplier_multiplier * 1000  # Scaling factor so it aligns with your millions-scale data
)

# 6. Clipping boundaries
df["production_impact_pct"] = df["production_impact_pct"].clip(0, 100)
df["full_recovery_days"] = df["full_recovery_days"].clip(1, 365)
print(df.shape)

(100000, 15)


In [14]:
# for production impact

features_impact = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "disruption_severity",
    "response_type",
    "response_time_days"
]

y = df["production_impact_pct"]

In [15]:
features_recovery = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "disruption_severity",
    "response_type",
    "response_time_days"
]

y = df["full_recovery_days"]

In [16]:
print(df["supplier_tier"].dtype)

int64


In [17]:
print(df[["response_time_days", "production_impact_pct"]].corr(numeric_only=True))
print(df[["response_time_days", "full_recovery_days"]].corr(numeric_only=True))

                       response_time_days  production_impact_pct
response_time_days               1.000000               0.139267
production_impact_pct            0.139267               1.000000
                    response_time_days  full_recovery_days
response_time_days            1.000000            0.196041
full_recovery_days            0.196041            1.000000


In [18]:
features_revenue = [
    "production_impact_pct",
    "full_recovery_days",
    "disruption_severity",
    "response_time_days",
    "industry",
    "supplier_tier",
    "supplier_size"
]

y = df["revenue_loss_usd"]

In [19]:
print(df[[
    "production_impact_pct",
    "full_recovery_days",
    "revenue_loss_usd"
]].corr(numeric_only=True))

                       production_impact_pct  full_recovery_days  \
production_impact_pct               1.000000            0.351935   
full_recovery_days                  0.351935            1.000000   
revenue_loss_usd                    0.704187            0.780999   

                       revenue_loss_usd  
production_impact_pct          0.704187  
full_recovery_days             0.780999  
revenue_loss_usd               1.000000  


In [20]:
from sklearn.ensemble import HistGradientBoostingRegressor

impact_features = features_impact

impact_cat_cols = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "response_type"
]

impact_cat_cols = [c for c in impact_cat_cols if c in impact_features]

impact_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), impact_cat_cols)
], remainder="passthrough")

X = df[impact_features]
y = df["production_impact_pct"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

impact_model = Pipeline([
    ("preprocessor", impact_preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=42
    ))
])

impact_model.fit(X_train, y_train)
pred = impact_model.predict(X_test)

print("production Impact MAE:", mean_absolute_error(y_test, pred))
print("production Impact R2:", r2_score(y_test, pred))

joblib.dump(impact_model, "impact_model.pkl")

production Impact MAE: 7.754241945533765
production Impact R2: 0.7817644820618648


['impact_model.pkl']

In [21]:
recovery_features = features_recovery

recovery_cat_cols = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "response_type"
]

recovery_cat_cols = [c for c in recovery_cat_cols if c in recovery_features]

recovery_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), recovery_cat_cols)
], remainder="passthrough")

X = df[recovery_features]
y = np.log1p(df["full_recovery_days"])   # IMPORTANT FIX

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

recovery_model = Pipeline([
    ("preprocessor", recovery_preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=600,
        random_state=42
    ))
])

recovery_model.fit(X_train, y_train)

pred_log = recovery_model.predict(X_test)

pred = np.expm1(pred_log)
y_true = np.expm1(y_test)

print("RECOVERY MAE:", mean_absolute_error(y_true, pred))
print("RECOVERY R2:", r2_score(y_true, pred))

joblib.dump(recovery_model, "recovery_model.pkl")

RECOVERY MAE: 30.35431715270984
RECOVERY R2: 0.3827769350406569


['recovery_model.pkl']

In [22]:
revenue_features = features_revenue

revenue_cat_cols = [
    "disruption_type",
    "industry",
    "supplier_tier",
    "supplier_region",
    "supplier_size",
    "has_backup_supplier",
    "response_type"
]

revenue_cat_cols = [c for c in revenue_cat_cols if c in revenue_features]

revenue_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), revenue_cat_cols)
], remainder="passthrough")

q99 = df["revenue_loss_usd"].quantile(0.99)
df_rev = df[df["revenue_loss_usd"] <= q99]

X = df_rev[revenue_features]
y = np.log1p(df_rev["revenue_loss_usd"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

revenue_model = Pipeline([
    ("preprocessor", revenue_preprocessor),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=10,
        max_iter=700,
        random_state=42
    ))
])

revenue_model.fit(X_train, y_train)

pred_log = revenue_model.predict(X_test)

pred = np.expm1(pred_log)
y_true = np.expm1(y_test)

print("REVENUE MAE:", mean_absolute_error(y_true, pred))
print("REVENUE R2:", r2_score(y_true, pred))

joblib.dump(revenue_model, "revenue_model.pkl")

REVENUE MAE: 55150.919344503964
REVENUE R2: 0.9986935812497042


['revenue_model.pkl']

In [23]:
print(df["revenue_loss_usd"].describe())

count    1.000000e+05
mean     4.773940e+06
std      5.504937e+06
min      8.517188e+03
25%      1.267845e+06
50%      2.977078e+06
75%      6.223445e+06
max      1.039024e+08
Name: revenue_loss_usd, dtype: float64


In [24]:
sample = pd.DataFrame(
    {
        "disruption_type":["Geopolitical"],
        "industry":["Automotive"],
        "supplier_tier":["Tier 1"],
        "supplier_region":["East Asia"],
        "supplier_size":["Large"],
        "has_backup_supplier":["No"],
        "disruption_severity":[5],
        "response_type":["Alternative Supplier"],
        "response_time_days":[10]
    }
)

In [25]:
predicted_impact = impact_model.predict(
    sample
)[0]

In [26]:
predicted_recovery = recovery_model.predict(
    sample
)[0]

In [27]:
revenue_input = pd.DataFrame(
    {
        "production_impact_pct":[predicted_impact],
        "full_recovery_days":[predicted_recovery],
        "disruption_severity":[5],
        "response_time_days":[10],
         "supplier_size":["Large"],
          "industry":["Automotive"],
          "supplier_tier":["Tier 1"]
    }
)

predicted_revenue = revenue_model.predict(
    revenue_input
)[0]

In [28]:
def calculate_risk(
    impact,
    recovery,
    revenue
):

    impact_norm = impact / 100

    recovery_norm = min(
        recovery / 365,
        1
    )

    revenue_norm = min(
        revenue / 10000000,
        1
    )

    risk_score = (
        0.5 * impact_norm
        +
        0.3 * recovery_norm
        +
        0.2 * revenue_norm
    ) * 100

    return risk_score

In [29]:
def risk_level(score):

    if score < 25:
        return "Low"

    elif score < 50:
        return "Medium"

    elif score < 75:
        return "High"

    else:
        return "Critical"

In [30]:
risk_score = calculate_risk(
    predicted_impact,
    predicted_recovery,
    predicted_revenue
)

risk = risk_level(
    risk_score
)

In [31]:
print(
    "Predicted Production Impact:",
    round(predicted_impact,2),
    "%"
)

print(
    "Predicted Recovery Time:",
    round(predicted_recovery,2),
    "days"
)

print(
    "Predicted Revenue Loss: $",
    round(predicted_revenue,2)
)

print(
    "Risk Score:",
    round(risk_score,2)
)

print(
    "Risk Level:",
    risk
)

Predicted Production Impact: 45.57 %
Predicted Recovery Time: 4.78 days
Predicted Revenue Loss: $ 13.14
Risk Score: 23.18
Risk Level: Low


In [32]:

suppliers = pd.read_csv(
    "../data/supplier_dataset.csv"
)

In [33]:
country_region = {

    "India":"South Asia",
    "Vietnam":"South East Asia",
    "Thailand":"South East Asia",
    "Malaysia":"South East Asia",
    "Singapore":"South East Asia",
    "Indonesia":"South East Asia",

    "Germany":"Europe",
    "France":"Europe",
    "Poland":"Europe",
    "Turkey":"Europe",

    "USA":"North America",
    "Canada":"North America",
    "Mexico":"North America",

    "Brazil":"South America"

}

suppliers["region"] = suppliers[
    "country"
].map(country_region)

In [34]:
def recommend_suppliers(
    current_region,
    current_tier,
    risk_level
):

    candidates = suppliers.copy()

    # same supplier tier

    candidates = candidates[
        candidates["supplier_tier"]
        == current_tier
    ]

    # avoid risky region

    if risk_level in ["High","Critical"]:

        candidates = candidates[
            candidates["region"]
            != current_region
        ]

    # scoring

    candidates["score"] = (

        candidates["supplier_rating"] * 40

        +

        candidates["quality_score"] * 0.6

        +

        (100 - candidates["cost_index"]) * 0.2

        +

        (15 - candidates["lead_time_days"]) * 2

        +

        candidates["backup_capability"].map(
            {
                "Yes":20,
                "No":0
            }
        )
    )

    candidates = candidates.sort_values(
        by="score",
        ascending=False
    )

    return candidates[
        [
            "supplier_id",
            "country",
            "supplier_rating",
            "lead_time_days",
            "quality_score",
            "score"
        ]
    ].head(5)

In [35]:
predicted_impact = 42
predicted_recovery = 75
predicted_revenue = 2200000

risk_score = calculate_risk(
    predicted_impact,
    predicted_recovery,
    predicted_revenue
)

risk = risk_level(
    risk_score
)

In [36]:
current_region = "East Asia"

current_tier = "Tier 1"

In [37]:
recommendations = recommend_suppliers(
    current_region=current_region,
    current_tier=current_tier,
    risk_level=risk
)

print(recommendations)

   supplier_id      country  supplier_rating  lead_time_days  quality_score  \
14      SUP015    Singapore              4.8               6             95   
18      SUP019          USA              4.8               7             95   
4       SUP005        Japan              4.8              10             93   
17      SUP018       Canada              4.7               8             93   
13      SUP014  South Korea              4.7               7             91   

    score  
14  291.8  
18  290.0  
4   281.8  
17  281.8  
13  281.8  


In [40]:
print("\n----- AI RISK ASSESSMENT -----\n")

print(
    "Production Impact:",
    round(predicted_impact,2),
    "%"
)

print(
    "Recovery Time:",
    round(predicted_recovery,2),
    "days"
)

print(
    "Revenue Loss: $",
    round(predicted_revenue,2)
)

print(
    "Risk Score:",
    round(risk_score,2)
)

print(
    "Risk Level:",
    risk
)

print("----- RECOMMENDED SUPPLIERS -----\n")

print(recommendations)


----- AI RISK ASSESSMENT -----

Production Impact: 42 %
Recovery Time: 75 days
Revenue Loss: $ 2200000
Risk Score: 31.56
Risk Level: Medium
----- RECOMMENDED SUPPLIERS -----

   supplier_id      country  supplier_rating  lead_time_days  quality_score  \
14      SUP015    Singapore              4.8               6             95   
18      SUP019          USA              4.8               7             95   
4       SUP005        Japan              4.8              10             93   
17      SUP018       Canada              4.7               8             93   
13      SUP014  South Korea              4.7               7             91   

    score  
14  291.8  
18  290.0  
4   281.8  
17  281.8  
13  281.8  
